In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [12]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task='text-generation'
)

In [13]:
model = ChatHuggingFace(llm = llm)

parser = StrOutputParser()

In [14]:
# state creation

class BlogState(TypedDict):

    title: str
    outline: str
    content: str


In [24]:
# create outline function

def create_outline(state: BlogState) -> BlogState:

    # extract title from the state
    title = state['title']

    # create a template
    template = PromptTemplate(
        template='Generate a detailed outline (proper structured) for a blog on the topic - {title}',
        input_variables=['title']
    )

    # create prompt
    prompt = template.invoke({"title": title})

    # call llm to gen outline
    gen_outline = model.invoke(prompt).content

    # update the outline in the state
    state['outline'] = gen_outline

    # return state
    return state

In [25]:
# create_blog function

def create_blog(state: BlogState) -> BlogState :

    # extract title and outline from the state
    title = state['title']
    outline = state['outline']

    # create a template
    template = PromptTemplate(
        template='Generate a detailed blog on the title - {title} using the following outline \n{outline}',
        input_variables=['title', 'outline']
    )

    # create prompt
    prompt = template.invoke({"title": title, 'outline': outline})

    # call llm to gen outline
    gen_content = model.invoke(prompt).content

    # update the outline in the state
    state['content'] = gen_content

    # return state
    return state
    

In [26]:
# graph creation
graph = StateGraph(BlogState)

# add nodes to the graph
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# add edges to the graph
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

# compile graph
workflow = graph.compile()

In [27]:
# execute the graph
initial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)
print(final_state['title'])
print(final_state['outline'])
print(final_state['content'])


{'title': 'Rise of AI in India', 'outline': 'Here is a detailed outline for a blog on the topic "Rise of AI in India":\n\n**I. Introduction**\n\n* Brief overview of the increasing importance of AI in India\n* Thesis statement: The rise of AI in India is transforming various industries and revolutionizing the way businesses operate.\n\n**II. Current State of AI in India**\n\n* Overview of the current AI landscape in India, including:\n\t+ Investment in AI research and development\n\t+ Government initiatives and policies supporting AI adoption\n\t+ Key players and startups in the Indian AI ecosystem\n* Statistics and facts to highlight the growth and adoption of AI in India\n\n**III. Applications of AI in Indian Industries**\n\n* A sector-by-sector analysis of AI applications in India, including:\n\t+ Healthcare: AI-powered diagnosis, personalized medicine, and medical research\n\t+ Finance: AI-driven customer service, risk management, and credit scoring\n\t+ Education: AI-powered learni